# ActionPulse diarization worker on a free GPU (Kaggle or Colab)

Runs `diarization-worker/app.py` (faster-whisper + pyannote community-1) on the notebook's GPU and
exposes it through a free Cloudflare quick tunnel. Paste the printed URL into Render as `DIARIZATION_WORKER_URL`.

**Before Run All**
- **Kaggle:** Settings → *Accelerator* = **GPU T4 x2**, *Internet* = **On**. Add-ons → *Secrets*: `HF_TOKEN`, `WORKER_SECRET` (tick both for this notebook).
- **Colab:** Runtime → *Change runtime type* → **T4 GPU**. 🔑 *Secrets*: `HF_TOKEN`, `WORKER_SECRET` with notebook access on.
- Accept the terms at https://huggingface.co/pyannote/speaker-diarization-community-1 with the account that owns `HF_TOKEN`.

The red *"pip's dependency resolver…"* text in step 1 is harmless: it's about unrelated preinstalled packages.
Keep the notebook running while you use the app. The tunnel URL changes every run, so update Render each time.

In [ ]:
# 1. Install (2-4 min). nvidia-cublas/cudnn give faster-whisper the CUDA 12 libraries it needs on the GPU.
!pip install -q faster-whisper "pyannote.audio>=4.0.0" fastapi "uvicorn[standard]" python-multipart nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)

In [ ]:
# 2. Fetch the worker (cache-busted so you always get the latest) + cloudflared
import time, subprocess
BRANCH = "claude/blissful-tesla-oybu12"  # switch to "main" once the PR is merged
url = f"https://raw.githubusercontent.com/Neel7780/AI-Powered-Meeting-to-Action-Converter/{BRANCH}/diarization-worker/app.py?nocache={int(time.time())}"
subprocess.run(["wget", "-q", "-O", "app.py", url], check=True)
assert "/selftest" in open("app.py").read(), "Got an old app.py; re-run this cell in a minute"
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Secrets (Kaggle Secrets or Colab userdata)
import os
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        from google.colab import userdata
        return userdata.get(name)

os.environ["HF_TOKEN"] = secret("HF_TOKEN")
os.environ["WORKER_SECRET"] = secret("WORKER_SECRET")
os.environ["WHISPER_MODEL"] = "small"   # "medium" is more accurate and still fits a T4
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU is plenty
print("secrets loaded")

In [ ]:
# 4. Start the worker with the CUDA 12 libraries on LD_LIBRARY_PATH, then run the self-test
import os, sys, time, requests, subprocess, importlib.util

def pkg_lib(pkg):
    spec = importlib.util.find_spec(pkg)
    return os.path.join(list(spec.submodule_search_locations)[0], "lib") if spec else None

libs = [p for p in (pkg_lib("nvidia.cublas"), pkg_lib("nvidia.cudnn")) if p and os.path.isdir(p)]
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = ":".join(libs + [env.get("LD_LIBRARY_PATH", "")])
print("CUDA libs:", libs)

if "worker" in globals() and worker.poll() is None:
    worker.terminate(); worker.wait()
log = open("worker.log", "w")
worker = subprocess.Popen([sys.executable, "-u", "app.py"], stdout=log, stderr=subprocess.STDOUT, env=env)
for _ in range(60):
    try:
        requests.get("http://localhost:7860/", timeout=2); break
    except Exception:
        time.sleep(2)

print("Loading models + self-test (first time downloads ~1 GB, 1-3 min)...")
r = requests.get("http://localhost:7860/selftest", headers={"X-Worker-Secret": os.environ["WORKER_SECRET"]}, timeout=1200)
print(r.status_code, r.text)
if r.status_code != 200:
    time.sleep(1)
    print("\n----- worker.log -----\n" + open("worker.log").read()[-4000:])
    raise SystemExit("Self-test failed: the reason is printed above")
print(requests.get("http://localhost:7860/").json())  # device should be "cuda"

In [ ]:
# 5. Public HTTPS URL via Cloudflare quick tunnel (free, no account)
import re
if "tunnel" in globals() and tunnel.poll() is None:
    tunnel.terminate()
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:7860"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\nDIARIZATION_WORKER_URL =", m.group(0))
        print("Set it (and DIARIZATION_WORKER_SECRET = your WORKER_SECRET) on Render, then redeploy.")
        break

In [ ]:
# 6. Keep alive + show worker logs (stop this cell to shut down)
while worker.poll() is None:
    time.sleep(60)
    print(open("worker.log").read()[-1500:], flush=True)
print("worker exited:", worker.returncode)